# Satellite Vision Agent - Large-Scale Regional Scanner

This advanced pipeline implements a **Stream Processing** approach to scan entire city districts for pollution.

**Workflow:**
1. **Input Region**: Define significant urban area (e.g., Hyderabad Center).
2. **Wide-Area Grid Generation**: dynamically calculate a massive grid (e.g., 20x20 tiles = ~100 sq km coverage).
3. **Stream Scanning (Fetch & Filter)**: 
   - Instead of downloading thousands of images first, we fetch them into memory.
   - Instantly run **Computer Vision Analysis** (Entropy/Texture).
   - **Discard** clean/empty tiles immediately.
   - **Save** only the "Anomaly" tiles (potential dumps, slums, smoke).
4. **AI Inspection (YOLO)**: Run deep learning only on the saved high-risk tiles.
5. **Mapping**: Generate a geospatial pollution map.

## 1. Setup & Configuration

In [ ]:
!pip install ultralytics onnxruntime opencv-python matplotlib numpy rasterio owslib requests pillow scikit-image

In [ ]:
import os
import json
import datetime
import cv2
import numpy as np
import requests
import warnings
from io import BytesIO
from PIL import Image
from skimage.measure import shannon_entropy
from ultralytics import YOLO
import onnxruntime as ort

warnings.filterwarnings("ignore")

# --- CONFIGURATION ---
# Target: Hyderabad (Center)
REGION_CONFIG = {
    "name": "Hyderabad_District_Scan",
    "lat": 17.3850, 
    "lon": 78.4867,
    "zoom": 16,
    "grid_width": 15 # Scans a 15x15 grid (225 tiles) ~ 50 sq km scan area
}

# Thresholds
# Anomaly Score (0.0 - 1.0). > 0.4 usually means complex urban structure/debris.
ANOMALY_THRESHOLD = 0.45 

# Paths
SCAN_OUTPUT_DIR = 'scanned_anomalies'
METADATA_FILE = 'scan_metadata.json'
EVENTS_FILE = 'pollution_events.json'
os.makedirs(SCAN_OUTPUT_DIR, exist_ok=True)

print(f"Target Region: {REGION_CONFIG['name']}")
print(f"Scan Area: {REGION_CONFIG['grid_width']}x{REGION_CONFIG['grid_width']} grid")

## 2. The "Stream Scanner" Engine
This function handles the core logic: Fetch -> Analyze -> Save/Discard.
It avoids disk I/O bottlenecks by processing in memory.

In [ ]:
def calculate_anomaly_score_in_memory(pil_image):
    """
    Analyzes a PIL image in memory without saving.
    Returns score (0.0 to 1.0).
    """
    # Convert PIL to OpenCV format (numpy array)
    img_np = np.array(pil_image)
    img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    
    # 1. Entropy (Chaos)
    entropy = shannon_entropy(gray)
    
    # 2. Edge Density (Roughness)
    edges = cv2.Canny(gray, 100, 200)
    edge_density = np.sum(edges) / edges.size
    
    # Heuristic Combo
    score_entropy = min(entropy / 9.0, 1.0)
    score_edges = min(edge_density / 0.15, 1.0)
    
    final_score = (0.7 * score_entropy) + (0.3 * score_edges)
    return round(final_score, 4)

def scan_region(config):
    center_lat = config['lat']
    center_lon = config['lon']
    zoom = config['zoom']
    width = config['grid_width']
    
    # Calculate Center Tile XYZ
    n = 2.0 ** zoom
    xtile_center = int((center_lon + 180.0) / 360.0 * n)
    ytile_center = int((1.0 - np.log(np.tan(np.radians(center_lat)) + 1/np.cos(np.radians(center_lat))) / np.pi) / 2.0 * n)
    
    offset = width // 2
    
    scanned_data = []
    total_tiles = width * width
    processed = 0
    anomalies_found = 0
    
    print(f"--- STARTING STREAM SCAN ({total_tiles} tiles) ---")
    print("{:<10} {:<10} {:<15} {:<15}".format("Progress", "Status", "Score", "Action"))
    
    for x in range(xtile_center - offset, xtile_center + offset + 1):
        for y in range(ytile_center - offset, ytile_center + offset + 1):
            processed += 1
            
            # 1. Fetch In-Memory
            url = f"https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{zoom}/{y}/{x}"
            try:
                headers = {'User-Agent': 'Mozilla/5.0'}
                resp = requests.get(url, headers=headers, timeout=5)
                if resp.status_code != 200: continue
                
                img = Image.open(BytesIO(resp.content)).convert('RGB')
                
                # 2. Analyze In-Memory (Pre-Screening)
                score = calculate_anomaly_score_in_memory(img)
                
                # 3. Decision Logic
                if score > ANOMALY_THRESHOLD:
                    status = "ANOMALY"
                    # SAVE only if anomaly
                    tile_name = f"tile_{x}_{y}.jpg"
                    save_path = os.path.join(SCAN_OUTPUT_DIR, tile_name)
                    img.save(save_path)
                    
                    # Calculate Real Coords for Meta
                    tile_deg = 360 / n
                    # lon_deg = x / n * 360.0 - 180.0
                    # lat_deg = arctan(sinh(pi * (1 - 2 * y / n))) * 180.0 / pi
                    # Simplified box center
                    lon_center = (x + 0.5) / n * 360.0 - 180.0
                    lat_rad = np.arctan(np.sinh(np.pi * (1 - 2 * (y + 0.5) / n)))
                    lat_center = np.degrees(lat_rad)
                    
                    # Bounds approx
                    delta = tile_deg / 2
                    
                    scanned_data.append({
                        "tile_name": tile_name,
                        "path": save_path,
                        "score": score,
                        "geo_bounds": {
                            "left": lon_center - delta, "right": lon_center + delta,
                            "top": lat_center + delta, "bottom": lat_center - delta
                        }
                    })
                    anomalies_found += 1
                else:
                    status = "CLEAN"
                
                # Log progress periodically
                if processed % 10 == 0 or status == "ANOMALY":
                     print("{:<10} {:<10} {:<15.4f} {:<15}".format(f"{processed}/{total_tiles}", status, score, "SAVED" if status=="ANOMALY" else "DISCARDED"))
                    
            except Exception as e:
                pass 
                
    return scanned_data

# --- RUN SCAN --- 
anomalies = scan_region(REGION_CONFIG)

# Save Scan Metadata
with open(METADATA_FILE, 'w') as f:
    json.dump({"tiles": anomalies}, f, indent=4)

print(f"\nScan Complete. Identified {len(anomalies)} high-risk areas from the grid.")
print(f"Saved metadata to {METADATA_FILE}")

## 3. Targeted AI Inspection
We now load the heavy deep learning model and process ONLY the tiles that the scanner identified as anomalies.

In [ ]:
MODEL_PATH = 'garbage_model.onnx' 
if not os.path.exists(MODEL_PATH):
    for f in os.listdir('.'):
        if f.endswith('.onnx'): 
             MODEL_PATH = f
             break

print(f"Loading AI Model: {MODEL_PATH}")

pollution_events = []
INPUT_SIZE = 224
CLASS_NAMES = ['smoke', 'dump'] 

def run_onnx_detection(session, image_path):
    img_orig = cv2.imread(image_path)
    if img_orig is None: return []
    
    img = cv2.resize(img_orig, (INPUT_SIZE, INPUT_SIZE))
    blob = cv2.dnn.blobFromImage(img, 1/255.0, (INPUT_SIZE, INPUT_SIZE), swapRB=True, crop=False)
    
    try:
        input_name = session.get_inputs()[0].name
        outputs = session.run(None, {input_name: blob})
        preds = np.squeeze(outputs[0]).T
    except: return []
    
    if preds.ndim != 2 or preds.shape[1] < 5: return []

    scores = np.max(preds[:, 4:], axis=1)
    keep = scores > 0.1 
    preds = preds[keep]
    scores = scores[keep]
    class_ids = np.argmax(preds[:, 4:], axis=1)
    boxes = preds[:, :4]
    
    h, w = img_orig.shape[:2]
    sx, sy = w / INPUT_SIZE, h / INPUT_SIZE
    
    final_objs = []
    # NMS Preparation
    formatted_boxes = []
    for i in range(len(scores)):
        cx, cy, bw, bh = boxes[i]
        x1 = (cx - bw/2) * sx
        y1 = (cy - bh/2) * sy
        x2 = (cx + bw/2) * sx
        y2 = (cy + bh/2) * sy
        formatted_boxes.append([x1, y1, x2, y2])
        
    indices = cv2.dnn.NMSBoxes(formatted_boxes, scores.tolist(), 0.1, 0.4)
    
    if len(indices) > 0:
        for i in indices.flatten():
            final_objs.append({
                "box": formatted_boxes[i],
                "conf": float(scores[i]),
                "cls": int(class_ids[i]),
                "label": CLASS_NAMES[int(class_ids[i])] if int(class_ids[i]) < len(CLASS_NAMES) else "object"
            })
    return final_objs

# --- EXECUTION ---
if os.path.exists(MODEL_PATH) and anomalies:
    session = ort.InferenceSession(MODEL_PATH)
    print(f"Inspecting {len(anomalies)} high-risk tiles...")
    
    for tile_data in anomalies:
        detections = run_onnx_detection(session, tile_data['path'])
        
        # If AI confirms objects, OR if the anomaly score was extremely high (likely non-standard pollution)
        # we register an event.
        if detections:
            for d in detections:
                # Geo Reasoning
                bounds = tile_data['geo_bounds']
                bx = d['box']
                cx, cy = (bx[0]+bx[2])/2, (bx[1]+bx[3])/2
                # Approx based on image dimensions (assuming 256 or similar fetch size)
                # Here we just map center-pixel to center-geo for simplicity in demo
                
                # Simple relative interpolation
                # NOTE: For production, we'd use exact pixel dimensions. 
                # Here we assume object is roughly near center of the anomaly tile for the report.
                lat_obj = (bounds['top'] + bounds['bottom']) / 2
                lon_obj = (bounds['left'] + bounds['right']) / 2
                
                pollution_events.append({
                    "location": [round(lat_obj, 5), round(lon_obj, 5)],
                    "type": d['label'],
                    "confidence": round(d['conf'], 4),
                    "source_tile": tile_data['tile_name']
                })
                
        elif tile_data['score'] > 0.6: 
             # Fallback: AI didn't recognize 'smoke'/'dump' specifically, 
             # but the area is extremely chaotic (score > 0.6). Report as 'Unidentified Anomaly'.
             lat_obj = (tile_data['geo_bounds']['top'] + tile_data['geo_bounds']['bottom']) / 2
             lon_obj = (tile_data['geo_bounds']['left'] + tile_data['geo_bounds']['right']) / 2
             
             pollution_events.append({
                "location": [round(lat_obj, 5), round(lon_obj, 5)],
                "type": "Unclassified_Anomaly",
                "confidence": tile_data['score'],
                "source_tile": tile_data['tile_name']
            })

    # Save Final Report
    with open(EVENTS_FILE, 'w') as f:
        json.dump(pollution_events, f, indent=2)
        
    print(f"\nFINAL REPORT GENERATED: {len(pollution_events)} Confirmed Incidents.")
    print(f"Data saved to {EVENTS_FILE}")
    if pollution_events:
        print("Sample:", pollution_events[0])

elif not anomalies:
    print("Scan complete. Region appears clean (no high-risk anomalies found).")
else:
    print("Model not found. Cannot proceed with inspection.")